# Diabetes Prediction — Modeling & Evaluation

## Overview

This notebook focuses on building and evaluating machine learning models to predict diabetes status using the prepared CDC Diabetes Health Indicators dataset.

The objective is to establish baseline performance, compare different classification algorithms, evaluate their generalization using cross-validation, and improve the selected model through hyperparameter tuning.

## Objectives

- Load the prepared training and test datasets.
- Define preprocessing steps for different feature types.
- Train and compare baseline classification models.
- Evaluate model performance using stratified cross-validation.
- Tune Logistic Regression hyperparameters.
- Identify potential improvements for handling class imbalance.

## Evaluation Strategy

The positive class is `Diabetes_binary = 1`. Because the target is imbalanced, evaluation will focus on Precision, Recall, F1-score, and Average Precision rather than relying on Accuracy alone.

The test set should be reserved for final evaluation and should not be used to select models, hyperparameters, or classification thresholds.

## 1. Import Libraries and Load Prepared Data

This section imports the libraries required for preprocessing, model training, evaluation, cross-validation, and hyperparameter tuning.

The prepared dataset is loaded from the output of the previous notebook, `06_feature_engineering_preparation.ipynb`.

The target variable, `Diabetes_binary`, is stored separately from the input features.

We use the training set for model development and cross-validation, while the test set is intended for the final unbiased evaluation.

The dataset shapes are printed to verify that the files were loaded correctly.

In [1]:
import pandas as pd
from pathlib import Path

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.base import clone
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import (
    StratifiedKFold,
    cross_validate,
    GridSearchCV,
    train_test_split,
)
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    make_scorer,
)

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import RandomOverSampler, SMOTE
from imblearn.under_sampling import RandomUnderSampler

In [2]:
data_dir = Path("../data/processed/prepared")

X_train = pd.read_csv(data_dir / "X_train.csv")
X_test = pd.read_csv(data_dir / "X_test.csv")

y_train = pd.read_csv(data_dir / "y_train.csv")["Diabetes_binary"]
y_test = pd.read_csv(data_dir / "y_test.csv")["Diabetes_binary"]

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (183579, 21)
X_test: (45895, 21)
y_train: (183579,)
y_test: (45895,)


## 2. Feature Groups

The input features are grouped according to their characteristics:

- **Numeric features:** `BMI`, `MentHlth`, and `PhysHlth`.
- **Binary features:** variables containing two unique values.
- **Ordinal features:** `GenHlth`, `Education`, `Income`, and `Age`, which represent ordered categories.

This grouping helps determine which preprocessing operations are appropriate for the baseline models.

Binary and ordinal variables are retained in their existing encoded form. Numeric features are standardized for models that are sensitive to feature scale.

In [3]:
numeric_features = ["BMI", "MentHlth", "PhysHlth"]

binary_features = [
    col for col in X_train.columns
    if X_train[col].nunique() == 2
]

ordinal_features = [
    "GenHlth", "Education", "Income", "Age"
]

print("Numeric:", numeric_features)
print("Binary:", binary_features)
print("Ordinal:", ordinal_features)

Numeric: ['BMI', 'MentHlth', 'PhysHlth']
Binary: ['HighBP', 'HighChol', 'CholCheck', 'Smoker', 'Stroke', 'HeartDiseaseorAttack', 'PhysActivity', 'Fruits', 'Veggies', 'HvyAlcoholConsump', 'AnyHealthcare', 'NoDocbcCost', 'DiffWalk', 'Sex']
Ordinal: ['GenHlth', 'Education', 'Income', 'Age']


## 3. Preprocessing Pipeline

A `ColumnTransformer` is used to standardize the selected numeric features with `StandardScaler`. All remaining features are passed through unchanged.

Standardization is particularly useful for Logistic Regression and K-Nearest Neighbors because their behavior can be affected by differences in feature scales.

Decision Tree and Random Forest do not require feature scaling, so their baseline pipelines use the original feature values.

Keeping preprocessing inside a pipeline ensures that, during cross-validation, the scaler is fitted only on the training portion of each fold, helping prevent data leakage.

In [4]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
    ],
    remainder="passthrough"
)

## 4. Baseline Model Selection

Five baseline configurations are evaluated:

1. **Logistic Regression:** a linear classification baseline.
2. **Balanced Logistic Regression:** Logistic Regression with `class_weight="balanced"` to account for class imbalance.
3. **K-Nearest Neighbors (KNN):** predicts a class based on nearby training examples.
4. **Decision Tree:** learns decision rules from the input features.
5. **Random Forest:** combines multiple decision trees to improve predictive performance.

The models use their initial configurations rather than an extensive hyperparameter search. This provides a starting point for comparing their behavior before further optimization.

In [5]:
models = {
    "Logistic Regression": Pipeline([
        ("preprocessor", preprocessor),
        ("model", LogisticRegression(random_state=42))
    ]),
    "Logistic Regression Balanced": Pipeline([
        ("preprocessor", preprocessor),
        ("model", LogisticRegression(
            class_weight="balanced",
            random_state=42
        ))
    ]),
    "KNN": Pipeline([
        ("preprocessor", preprocessor),
        ("model", KNeighborsClassifier(n_neighbors=5))
    ]),
    "Decision Tree": Pipeline([
        ("preprocessor", "passthrough"),
        ("model", DecisionTreeClassifier(random_state=42))
    ]),
    "Random Forest": Pipeline([
        ("preprocessor", "passthrough"),
        ("model", RandomForestClassifier(
            random_state=42,
            n_jobs=-1
        ))
    ])
}

## 5. Baseline Evaluation

The baseline models are fitted on the training set and evaluated using the test set. The results include Accuracy, Precision, Recall, and F1-score for the positive class.

- **Precision** measures how many predicted positive cases are actually positive.
- **Recall** measures how many actual positive cases are correctly identified.
- **F1-score** balances Precision and Recall.
- **Accuracy** measures the proportion of all predictions that are correct.

Because the target is imbalanced, a high Accuracy score alone does not necessarily indicate strong detection of diabetes cases.

**Methodological note:** These test-set results are exploratory because the test set has already been used for initial model comparisons. Subsequent model selection and tuning should rely on training-set cross-validation. The existing test results should not be treated as a completely untouched final evaluation.

In [6]:
results = []

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1": f1_score(y_test, y_pred, zero_division=0)
    })

results_df = pd.DataFrame(results).sort_values(
    "F1", ascending=False
)

display(results_df)

/home/muhammad-ammar/miniforge3/envs/cdc/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/home/muhammad-ammar/miniforge3/envs/cdc/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-lear

,Model,Accuracy,Precision,Recall,F1
1,Logistic Regression Balanced,0.714326,0.318193,0.759510,0.448492
3,Decision Tree,0.773897,0.286441,0.320843,0.302668
2,KNN,0.833969,0.416412,0.213278,0.282080
4,Random Forest,0.843077,0.463905,0.167545,0.246180
0,Logistic Regression,0.850572,0.540473,0.153156,0.238677


### Baseline Evaluation — Interpretation

These initial test-set results show the trade-off created by class imbalance. The unweighted Logistic Regression model has relatively high Accuracy but low Recall for class `1`, while `class_weight="balanced"` substantially increases Recall and F1-score.

Because this test set was used for these early comparisons, treat the table as an exploratory baseline only. The later cross-validation results on the training set are the primary basis for comparing configurations. The test set is not fully untouched, which must be disclosed when reporting final results.

## 6. Stratified Cross-Validation

To compare the baseline models more reliably, we use five-fold `StratifiedKFold` cross-validation on the training set.

Stratification preserves approximately the same class proportions across folds, which is important when the target is imbalanced.

The data is shuffled before splitting, and `random_state=42` makes the split reproducible.

Each model is evaluated using:

- **Recall:** the proportion of positive cases detected.
- **F1-score:** the balance between Precision and Recall.
- **Average Precision:** summarizes the precision-recall ranking performance across different thresholds.

For each metric, the mean and standard deviation across folds are reported. The mean describes average performance, while the standard deviation indicates variation across folds.

In [7]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [8]:
scoring = {
    "recall": make_scorer(recall_score, pos_label=1),
    "f1": make_scorer(f1_score, pos_label=1),
    "average_precision": "average_precision",
}

cv_results = []

for name, model in models.items():
    scores = cross_validate(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        n_jobs=-1
    )

    cv_results.append({
        "Model": name,
        "Recall Mean": scores["test_recall"].mean(),
        "Recall Std": scores["test_recall"].std(),
        "F1 Mean": scores["test_f1"].mean(),
        "F1 Std": scores["test_f1"].std(),
        "Average Precision Mean": scores["test_average_precision"].mean(),
        "Average Precision Std": scores["test_average_precision"].std(),
    })

cv_results_df = pd.DataFrame(cv_results).sort_values(
    "F1 Mean",
    ascending=False
)

display(cv_results_df.round(3))

/home/muhammad-ammar/miniforge3/envs/cdc/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/home/muhammad-ammar/miniforge3/envs/cdc/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-lear

,Model,Recall Mean,Recall Std,F1 Mean,F1 Std,Average Precision Mean,Average Precision Std
1,Logistic Regression Balanced,0.754,0.007,0.450,0.001,0.404,0.005
3,Decision Tree,0.320,0.008,0.301,0.006,0.196,0.002
2,KNN,0.208,0.005,0.276,0.006,0.283,0.002
4,Random Forest,0.163,0.003,0.243,0.006,0.363,0.004
0,Logistic Regression,0.148,0.004,0.232,0.005,0.406,0.005


### Cross-Validation Findings

Balanced Logistic Regression achieved the strongest mean F1-score and Recall among the evaluated baseline models.

Its mean F1-score was approximately `0.450`, with a mean Recall of approximately `0.754`. This indicates that class weighting substantially improves the detection of positive cases compared with the unweighted Logistic Regression baseline.

Unweighted Logistic Regression achieved a slightly higher Average Precision of approximately `0.406`, compared with approximately `0.404` for its balanced counterpart. This suggests that ranking performance and classification performance at the default threshold can favor different configurations.

These results support further investigation of class imbalance and classification thresholds. They do not yet establish a final model.

## 7. Logistic Regression — Hyperparameter Tuning

Balanced Logistic Regression performed well during baseline cross-validation, so it is selected for further investigation.

`GridSearchCV` evaluates combinations of the following parameters:

- `C`: controls the strength of regularization. Smaller values apply stronger regularization.
- `class_weight`: compares the default class weighting with `balanced`.
- `solver`: kept fixed at `lbfgs` to isolate the effects of the other parameters.

The search uses five-fold stratified cross-validation on the training set.

F1-score is the primary selection metric, while Recall and Average Precision are also recorded for comparison. Setting `refit="f1"` instructs GridSearchCV to refit the best configuration according to mean cross-validated F1-score.

The Logistic Regression estimator uses `max_iter=2000` to allow additional iterations during optimization.

In [9]:
logistic_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", LogisticRegression(max_iter=2000, random_state=42))
    ]
)

param_grid = {
    "model__C": [0.01, 0.1, 1, 10],
    "model__class_weight": [None, "balanced"],
    "model__solver": ["lbfgs"]
}

In [10]:
grid_search = GridSearchCV(
    estimator=logistic_pipeline,
    param_grid=param_grid,
    scoring={
        "f1": "f1",
        "recall": "recall",
        "average_precision": "average_precision"
    },
    refit="f1",
    cv=cv,
    n_jobs=-1,
    return_train_score=False
)

In [11]:
grid_search.fit(X_train, y_train)

/home/muhammad-ammar/miniforge3/envs/cdc/lib/python3.12/site-packages/joblib/externals/loky/process_executor.py:787: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'model__C': [0.01, 0.1, ...], 'model__class_weight': [None, 'balanced'], 'model__solver': ['lbfgs']}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.","{'average_precision': 'average_precision', 'f1': 'f1', 'recall': 'recall'}"
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",'f1'
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value 

In [12]:
results_df = pd.DataFrame(grid_search.cv_results_)

results_df = results_df[
    [
        "param_model__C",
        "param_model__class_weight",
        "mean_test_f1",
        "std_test_f1",
        "mean_test_recall",
        "mean_test_average_precision"
    ]
].sort_values("mean_test_f1", ascending=False)

display(results_df.round(3))

,param_model__C,param_model__class_weight,mean_test_f1,std_test_f1,mean_test_recall,mean_test_average_precision
5,1.00,balanced,0.449,0.001,0.754,0.404
3,0.10,balanced,0.449,0.002,0.754,0.404
7,10.00,balanced,0.449,0.001,0.754,0.404
1,0.01,balanced,0.449,0.002,0.753,0.404
4,1.00,NaN,0.233,0.005,0.148,0.406
6,10.00,NaN,0.233,0.005,0.148,0.406
2,0.10,NaN,0.232,0.005,0.148,0.406
0,0.01,NaN,0.227,0.006,0.144,0.405


In [13]:
print("Best Parameters:", grid_search.best_params_)
print("Best CV F1:", grid_search.best_score_)

Best Parameters: {'model__C': 1, 'model__class_weight': 'balanced', 'model__solver': 'lbfgs'}
Best CV F1: 0.4494718286310978


### Hyperparameter Tuning Results

The best configuration reported by the search was:

- `C = 1`
- `class_weight = "balanced"`
- `solver = "lbfgs"`

The best mean cross-validated F1-score was approximately `0.4495`.

The tested balanced configurations produced nearly identical F1-scores, Recall values, and Average Precision values. Therefore, the search did not demonstrate a meaningful improvement over the previous balanced Logistic Regression baseline.

The results suggest that changing `C` within the tested range had little effect on the reported metrics. Further improvements may require investigating class-imbalance strategies, feature representations, or other model families rather than expanding this same parameter grid without a clear hypothesis.

## 8. Class Imbalance Handling
The target variable is imbalanced, with approximately 84.7% of samples belonging to class 0 and 15.3% belonging to class 1.

The initial experiments showed that using `class_weight="balanced"` improved the Recall and F1-score of Logistic Regression. In this section, additional resampling strategies will be evaluated:

- Random Oversampling
- SMOTE
- Random Undersampling

All resampling operations will be performed inside the training folds using an imbalanced-learn pipeline. This prevents information from validation folds from influencing the resampling process.

The strategies will be compared using the same stratified cross-validation setup and evaluation metrics: F1-score, Recall, and Average Precision.

### 8.1 Resampling Pipelines

Each resampling strategy is integrated into a pipeline with Logistic Regression.

Random Oversampling duplicates minority-class samples, SMOTE generates synthetic minority-class samples, and Random Undersampling reduces the number of majority-class samples.

Resampling is applied only during pipeline fitting, ensuring that validation folds retain their original class distribution.

In [14]:
resampling_models = {
    "Balanced Logistic Regression": ImbPipeline([
        ("preprocessor", preprocessor),
        ("model", LogisticRegression(
            class_weight="balanced",
            max_iter=2000,
            random_state=42
        ))
    ]),
    "Random Oversampling": ImbPipeline([
        ("preprocessor", preprocessor),
        ("sampler", RandomOverSampler(random_state=42)),
        ("model", LogisticRegression(max_iter=2000, random_state=42))
    ]),
    "SMOTE": ImbPipeline([
        ("preprocessor", preprocessor),
        ("sampler", SMOTE(random_state=42)),
        ("model", LogisticRegression(max_iter=2000, random_state=42))
    ]),
    "Random Undersampling": ImbPipeline([
        ("preprocessor", preprocessor),
        ("sampler", RandomUnderSampler(random_state=42)),
        ("model", LogisticRegression(max_iter=2000, random_state=42))
    ])
}

In [15]:
scoring = {
    "f1": make_scorer(f1_score, pos_label=1),
    "recall": make_scorer(recall_score, pos_label=1),
    "average_precision": "average_precision"
}

resampling_results = []

for name, model in resampling_models.items():
    scores = cross_validate(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        n_jobs=-1
    )

    resampling_results.append({
        "Model": name,
        "F1 Mean": scores["test_f1"].mean(),
        "F1 Std": scores["test_f1"].std(),
        "Recall Mean": scores["test_recall"].mean(),
        "Average Precision Mean": scores["test_average_precision"].mean()
    })

resampling_results_df = (
    pd.DataFrame(resampling_results)
    .sort_values("F1 Mean", ascending=False)
)

display(resampling_results_df.round(3))

,Model,F1 Mean,F1 Std,Recall Mean,Average Precision Mean
0,Balanced Logistic Regression,0.449,0.001,0.754,0.404
1,Random Oversampling,0.449,0.001,0.754,0.404
3,Random Undersampling,0.449,0.001,0.755,0.404
2,SMOTE,0.448,0.002,0.748,0.401


### Resampling Comparison — Findings

Random Oversampling and Random Undersampling achieved nearly identical cross-validated performance to Balanced Logistic Regression. SMOTE produced slightly lower mean F1-score, Recall, and Average Precision in the current experiments.

These results do not show a meaningful improvement from resampling for Logistic Regression. Therefore, `class_weight="balanced"` remains a strong baseline due to its comparable performance and simpler training pipeline.

The next experiment will investigate whether resampling strategies provide greater benefits when combined with a different model family, such as Random Forest.

## 9. Random Forest with Class Imbalance Handling

The previous experiments showed that resampling did not meaningfully improve Logistic Regression. In this section, Random Forest is evaluated with different imbalance-handling strategies.

The experiments compare:

- Random Forest with `class_weight="balanced"`.
- Random Forest with Random Oversampling.
- Random Forest with SMOTE.
- Random Forest with Random Undersampling.

All configurations use the same stratified cross-validation strategy and evaluation metrics: F1-score, Recall, and Average Precision. Resampling is applied only within each training fold to prevent data leakage.

The objective is to determine whether these strategies improve minority-class detection for a tree-based ensemble model.

In [16]:
rf_resampling_models = {
    "RF Balanced": ImbPipeline([
        ("model", RandomForestClassifier(
            class_weight="balanced",
            random_state=42,
            n_jobs=1
        ))
    ]),
    "RF Oversampling": ImbPipeline([
        ("sampler", RandomOverSampler(random_state=42)),
        ("model", RandomForestClassifier(
            random_state=42,
            n_jobs=1
        ))
    ]),
    "RF SMOTE": ImbPipeline([
        ("sampler", SMOTE(random_state=42)),
        ("model", RandomForestClassifier(
            random_state=42,
            n_jobs=1
        ))
    ]),
    "RF Undersampling": ImbPipeline([
        ("sampler", RandomUnderSampler(random_state=42)),
        ("model", RandomForestClassifier(
            random_state=42,
            n_jobs=1
        ))
    ])
}

In [17]:
rf_results = []

for name, model in rf_resampling_models.items():
    scores = cross_validate(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        n_jobs=-1
    )

    rf_results.append({
        "Model": name,
        "F1 Mean": scores["test_f1"].mean(),
        "F1 Std": scores["test_f1"].std(),
        "Recall Mean": scores["test_recall"].mean(),
        "Average Precision Mean":
            scores["test_average_precision"].mean()
    })

rf_results_df = (
    pd.DataFrame(rf_results)
    .sort_values("F1 Mean", ascending=False)
)

display(rf_results_df.round(3))

,Model,F1 Mean,F1 Std,Recall Mean,Average Precision Mean
3,RF Undersampling,0.433,0.002,0.755,0.378
0,RF Balanced,0.407,0.003,0.422,0.351
1,RF Oversampling,0.348,0.005,0.298,0.345
2,RF SMOTE,0.266,0.002,0.189,0.353


### Random Forest Resampling — Findings

Random Forest with Random Undersampling achieved the highest mean F1-score (`0.433`) and Recall (`0.755`) among the tested Random Forest configurations.

Random Forest with `class_weight="balanced"` achieved a mean F1-score of `0.407` and Recall of `0.422`. Random Oversampling and SMOTE performed worse under the current experimental settings.

These findings suggest that Random Undersampling is a promising strategy for improving minority-class detection with Random Forest. However, its Average Precision (`0.378`) remains below the previously observed values for Logistic Regression.

The results are specific to the current models and settings. Further validation is required before selecting a final model.

## 10. Overall Model Comparison

The cross-validation experiments compared multiple baseline classifiers and class-imbalance handling strategies.

Balanced Logistic Regression achieved the highest mean F1-score among the evaluated configurations, at approximately `0.450`. Random Forest with Random Undersampling achieved a similar Recall (`0.755`) and a mean F1-score of approximately `0.433`.

Unweighted Logistic Regression achieved the highest observed Average Precision (`0.406`), although its default-threshold Recall and F1-score were substantially lower.

These results highlight the trade-off between ranking quality, positive-class detection, and the balance between Precision and Recall. The preferred model depends on the project's evaluation objective.

Balanced Logistic Regression and Random Forest with Random Undersampling will be considered the leading candidates for further investigation. Their performance should be compared under a consistent validation strategy before final model selection.

## 11. Validation-Based Threshold Selection

The current Balanced Logistic Regression model is the leading candidate based on cross-validated F1-score and Recall.

The previous threshold analysis identified `0.6` as the best F1-score among the tested thresholds. However, these results were exploratory and should not be treated as an independent estimate of performance.

In this section, a validation strategy will be used to select a classification threshold without using the test set. The selected threshold will then be evaluated on the test set only once, after the modeling decisions are finalized.

In [18]:
X_subtrain, X_val, y_subtrain, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.2,
    random_state=42,
    stratify=y_train
)

In [19]:
threshold_model = clone(grid_search.best_estimator_)

threshold_model.fit(X_subtrain, y_subtrain)

y_val_proba = threshold_model.predict_proba(X_val)[:, 1]

In [20]:
threshold_results = []

for threshold in [0.3, 0.4, 0.5, 0.6, 0.7]:
    y_val_pred = (y_val_proba >= threshold).astype(int)

    threshold_results.append({
        "Threshold": threshold,
        "Precision": precision_score(y_val, y_val_pred),
        "Recall": recall_score(y_val, y_val_pred),
        "F1-Score": f1_score(y_val, y_val_pred)
    })

threshold_df = pd.DataFrame(threshold_results)
display(threshold_df.round(3))

,Threshold,Precision,Recall,F1-Score
0,0.3,0.247,0.919,0.389
1,0.4,0.283,0.847,0.425
2,0.5,0.323,0.751,0.452
3,0.6,0.371,0.633,0.468
4,0.7,0.426,0.471,0.448


### Validation-Based Threshold Selection — Findings

The validation experiment evaluated classification thresholds from `0.3` to `0.7`.

Among the tested values, a threshold of `0.6` achieved the highest F1-score (`0.468`), with Precision of `0.371` and Recall of `0.633`.

Compared with the default threshold of `0.5`, the threshold of `0.6` improved F1-score and Precision but reduced Recall. This reflects the trade-off between identifying more positive cases and reducing false-positive predictions.

The threshold of `0.6` is selected as a candidate based on validation F1-score. Its suitability depends on the project's objective and must be distinguished from the final test-set performance.

## 12. Final Model Evaluation

The validation experiments identified Balanced Logistic Regression as the leading candidate, with a classification threshold of `0.6` achieving the highest validation F1-score among the tested thresholds.

In this section, the selected Logistic Regression configuration is refitted on the full training set. The model is then evaluated on the held-out test set using the fixed threshold of `0.6`.

The evaluation includes Precision, Recall, F1-score, Accuracy, and the confusion matrix.

**Important limitation:** The test set was used in earlier exploratory model comparisons. Therefore, these results should be interpreted as a final evaluation of the selected configuration, not as a completely untouched estimate of generalization performance.

In [21]:
final_model = clone(grid_search.best_estimator_)
final_model.fit(X_train, y_train)

y_test_proba = final_model.predict_proba(X_test)[:, 1]
y_test_pred = (y_test_proba >= 0.6).astype(int)

In [22]:
print("Accuracy:", accuracy_score(y_test, y_test_pred))
print("Average Precision:", average_precision_score(y_test, y_test_proba))
print("\nClassification Report:")
print(classification_report(y_test, y_test_pred, digits=3))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_test_pred))

Accuracy: 0.7749210153611504
Average Precision: 0.4121343066637563

Classification Report:
              precision    recall  f1-score   support

         0.0      0.925     0.799     0.857     38876
         1.0      0.366     0.642     0.466      7019

    accuracy                          0.775     45895
   macro avg      0.645     0.721     0.662     45895
weighted avg      0.840     0.775     0.798     45895

Confusion Matrix:
[[31058  7818]
 [ 2512  4507]]


### Final Evaluation Results

The selected Balanced Logistic Regression model was refitted on the full training set and evaluated on the held-out test set using a classification threshold of `0.6`.

The model achieved:

- Accuracy: `0.775`
- Positive-class Precision: `0.366`
- Positive-class Recall: `0.642`
- Positive-class F1-score: `0.466`
- Average Precision: `0.412`

The confusion matrix was:

`[[31058, 7818], [2512, 4507]]`

The model correctly identified 4,507 positive cases out of 7,019 actual positive cases, while missing 2,512 positive cases. It also produced 7,818 false-positive predictions.

These results demonstrate a trade-off between positive-case detection and false-positive predictions. The model should not be interpreted as a clinical diagnostic tool without further validation.

**Evaluation limitation:** The test set was used in earlier exploratory model comparisons. Therefore, these results are not a completely independent estimate of generalization performance.

## 13. Conclusion and Next Steps

### Conclusion

Balanced Logistic Regression was the strongest overall candidate in the current experiments based on cross-validated F1-score (`~0.450`) and Recall (`~0.754`). Random Forest with Random Undersampling achieved similar Recall (`~0.755`) but a lower mean F1-score (`~0.433`). Resampling methods did not meaningfully improve Logistic Regression in the tested configurations.

Using a validation split, threshold `0.6` achieved the highest F1-score among the tested thresholds (`0.468`). On the final evaluation run, the selected model achieved test F1-score `0.466`, Recall `0.642`, Precision `0.366`, and Average Precision `0.412`.

### Limitations

- The test set was used in early baseline comparisons, so the final test metrics are not a completely independent estimate of generalization performance.
- Only a limited set of thresholds and hyperparameters was evaluated.
- These models are for educational experimentation and should not be used for clinical diagnosis without appropriate clinical validation.

### Possible Next Steps

- If further improvement is needed, define the operational objective first (for example, higher Recall versus fewer false positives).
- Consider additional validation or nested cross-validation before making more model-selection decisions.
- If the project goal includes deployment, package the selected preprocessing and model pipeline and document the prediction limitations.